# WIP

In [2]:
import pandas as pd

df = pd.read_csv('data/train.csv')

In [3]:
df.head()

,ID,var3,var15,imp_ent_var16_ult1,imp_op_var39_comer_ult1,imp_op_var39_comer_ult3,imp_op_var40_comer_ult1,imp_op_var40_comer_ult3,imp_op_var40_efect_ult1,imp_op_var40_efect_ult3,...,saldo_medio_var33_hace2,saldo_medio_var33_hace3,saldo_medio_var33_ult1,saldo_medio_var33_ult3,saldo_medio_var44_hace2,saldo_medio_var44_hace3,saldo_medio_var44_ult1,saldo_medio_var44_ult3,var38,TARGET
0,1,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,39205.170000,0
1,3,2,34,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,49278.030000,0
2,4,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,67333.770000,0
3,8,2,37,0.0,195.0,195.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,64007.970000,0
4,10,2,39,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,117310.979016,0


In [9]:
from sklearn.metrics import roc_auc_score

res = {}

for f in df_train.columns:
    if f != 'TARGET':
        auc = roc_auc_score(df_train['TARGET'], df_train[f])
        res[f] = max(auc, 1-auc)

res = pd.Series(res).sort_values(ascending=False)

In [12]:
features = res[res > 0.51].index.tolist()

In [5]:
df_train['TARGET'].value_counts()

TARGET
0    73012
1     3008
Name: count, dtype: int64

In [ ]:
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import classification_report, roc_auc_score
import numpy as np


X = df_train[features]
y = df_train['TARGET']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)


scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

model = xgb.XGBClassifier(n_estimators=50, max_depth=4, learning_rate=0.1, eval_metric='logloss', scale_pos_weight=scale_pos_weight)
model.fit(X_train, y_train)

y_train_proba = model.predict_proba(X_train)[:, 1]
y_test_proba = model.predict_proba(X_test)[:, 1]

In [130]:
# print train and test roc_auc

# Calculate ROC AUC for Train
train_auc = roc_auc_score(y_train, y_train_proba)
print(f'Train ROC AUC (prob): {train_auc:.4f}')

# Calculate ROC AUC for Test
test_auc = roc_auc_score(y_test, y_test_proba)
print(f'Test ROC AUC (prob): {test_auc:.4f}')

Train ROC AUC (prob): 0.8665
Test ROC AUC (prob): 0.8266


In [ ]:
import pandas as pd
from sklearn.metrics import f1_score

from lightautoml.automl.presets.tabular_presets import TabularAutoML
from lightautoml.tasks import Task

df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')

automl = TabularAutoML(
    task = Task(
        name = 'binary',
        metric = lambda y_true, y_pred: f1_score(y_true, (y_pred > 0.5)*1))
)
oof_pred = automl.fit_predict(
    df_train,
    roles = {'target': 'TARGET'}
)
test_pred = automl.predict(df_test)

In [127]:
roc_auc_score(df_train['TARGET'], oof_pred._data[:, 0])

0.8378827682508616